# Imports

In [1]:
import os
import sys

from dataclasses import dataclass
from typing import Dict, Tuple, Any, Optional

import pandas as pd
import json
import geojson

import girder_client
import numpy as np

from shapely.geometry import Point
import uuid

import subprocess
from typing_extensions import Union

from __future__ import annotations

from fusion_tools.utils.shapes import load_visium, geojson_to_histomics

# Init Code

In [2]:
data_dir = "/home/anish.tatke/orange-group/anish.tatke/FUSION/Cell Deconvolution/"
item = "693854f7105c8d00c48fc2d2"

girderApiUrl = "https://devathena.rc.ufl.edu/api/v1/"
girderToken = "Mzj6pwHJCFqRirp4auiPHxooBJyVQ849ZYMzitOk3G6jpj3s0H1TJ21PxoOMvdZ7"

In [8]:
gc = girder_client.GirderClient(
    apiUrl=girderApiUrl
)
gc.setToken(girderToken)

In [3]:
spot_coords_path = None
if 'spot_coordinates.csv' in os.listdir(data_dir+'/'):
    print(f'Spot coordinates found at: {data_dir}/spot_coordinates.csv')
    spot_coords_path = f'{data_dir}/spot_coordinates.csv'

if not spot_coords_path is None:
    visium_spots = load_visium(spot_coords_path)

Spot coordinates found at: /home/anish.tatke/orange-group/anish.tatke/FUSION/Cell Deconvolution//spot_coordinates.csv
Finding MPP scale for 317 spots
MPP Found! 0.7591458644571283


Working on Spot: 317/317: 100%|██████████| 317/317 [00:00<00:00, 1156.30it/s]


# My Methods

In [ ]:
map_dict = os.path.join(data_dir, "Mapping_Materials", "Cell_SubTypes_Grouped.csv")
map_dict = pd.read_csv(map_dict)
map_dict.head()
# .to_dict('records')
# ref = {}
# for item in map_dict:
#     mt = item['Main_Types'] 
#     ref[mt] = item
#     ref[mt]['Sub_Types'] = ref[mt]['Sub_Types'].split('.')
#     ref[mt]['Cell_States'] = ref[mt]['Cell_States'].split('.')
#     assert len(ref[mt]['Sub_Types']) == len(ref[mt]['Cell_States'])
#     del ref[mt]['Main_Types']
    
# ref

In [ ]:
ref.keys()

In [ ]:
p1 = pd.read_csv(os.path.join(data_dir, 'predsubclassl1.csv'))
p1 = p1.rename(columns={"Unnamed: 0": "Main Celltype"})
p1.drop(p1.index[-1], inplace=True)
p1

In [ ]:
output_csvs = [i for i in os.listdir(data_dir+'/') if 'csv' in i and not i=='spot_coordinates.csv']
output_dict = {}
for o in output_csvs:
    if o in ['predsubclassl1.csv', 'pred_subclass_l1.csv']:
        output_dict["Main_Cell_Types"] = o
    elif o in ['predsubclassl2.csv', 'pred_subclass_l2.csv']:
        output_dict["L2_Cell_Types"] = o

print(output_dict)

In [ ]:
for s in visium_spots['features']:
    print(s['properties'])
    break

## Main Cell Type

In [ ]:
pl = pd.read_csv(os.path.join(data_dir, 'predsubclassl1.csv'))
pl = pl.rename(columns={"Unnamed: 0": "Main Celltype"})
main_cell_types = {}
for row in pl[["Main Celltype", spot_id]].iterrows():
    if row[1]["Main Celltype"] in map_dict["Main_Types"].to_list():
        main_cell_types[row[1]["Main Celltype"]] = row[1][spot_id]
    elif row[1]["Main Celltype"] == "max":
        continue
    else:
        print(row[1])
        
print(main_cell_types)
# for s,p in zip(visium_spots['features'], property_list):
#     s['properties'] = s['properties'] | p

In [ ]:
for s in visium_spots['features']:
    spot_id = s['properties']['barcode']
    for key, file in output_dict.items():
        pl = pd.read_csv(os.path.join(data_dir, file))
        pl = pl.rename(columns={"Unnamed: 0": key})
        if key == "Main_Cell_Types":
            main_cell_types = {}
            for row in pl[[key, spot_id]].iterrows():
                if row[1][key] in ref.keys():
                    main_cell_types[row[1][key]] = row[1][spot_id]
                elif row[1][key] == "max":
                    continue
            s['properties'] = s['properties'] | {key: main_cell_types}
        elif key == "L2_Cell_Types"
            

## L2 Cell Types

In [ ]:
for s in visium_spots['features']:
    spot_id = s['properties']['barcode']
    pl = pd.read_csv(os.path.join(data_dir, 'predsubclassl2.csv'))
    pl = pl.rename(columns={"Unnamed: 0": "L2_Celltype"})
    states = {}
    for row in pl[["L2_Celltype", spot_id]].iterrows():
        if row[1]["L2_Celltype"] in [type for val in ref.values() for type in val['Sub_Types']]:
            
        elif row[1]["L2_Celltype"] == "max":
            continue

In [ ]:
# cell_states_dict = {}
# for subtypes, cell_states in zip(map_dict["Sub_Types"].to_list(), map_dict["Cell_States"].to_list()):
#     subtypes = subtypes.split('.')
#     cell_states = cell_states.split('.')
#     assert len(subtypes) == len(cell_states)
    
#     for st, ct in zip(subtypes, cell_states):
#         cell_states_dict[st] = ct
# cell_states_dict

# Functional Code

In [4]:
@dataclass(frozen=True)
class RefMaps:
    main_types: set
    subtype_to_state: Dict[str, Tuple[str, str]]

In [10]:
def _read_celltype_x_spot_csv(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)

    first_col = df.columns[0]
    if str(first_col).startswith("Unnamed") or first_col in ("", None):
        df = df.rename(columns={first_col: "Main_Type"})

    if df.columns[0] != df.columns[1] and df.columns[0] not in df.columns[1:]:
        df = df.set_index(df.columns[0])

    df = df.apply(pd.to_numeric, errors="coerce").fillna(0.0)    
    df.drop(df.index[-1], inplace=True)
    df.index = df.index.astype(str)
    return df

def build_reference_maps(cell_subtypes_grouped_csv: str) -> RefMaps:
    ref = pd.read_csv(cell_subtypes_grouped_csv).fillna("")

    required = {"Main_Types", "Sub_Types", "Cell_States"}
    missing = required - set(ref.columns)
    if missing:
        raise ValueError(f"Reference CSV missing columns: {sorted(missing)}")

    main_types = set(ref["Main_Types"].astype(str).str.strip())

    subtype_to_state: Dict[str, Tuple[str, str]] = {}
    for _, row in ref.iterrows():
        main = str(row["Main_Types"]).strip()
        subtypes = [s.strip() for s in str(row["Sub_Types"]).split(".") if s.strip()]
        states = [s.strip() for s in str(row["Cell_States"]).split(".") if s.strip()]

        if not subtypes:
            continue
        if len(subtypes) != len(states):
            raise ValueError(
                f"Mismatch in reference for Main_Types='{main}': "
                f"{len(subtypes)} subtypes vs {len(states)} states.\n"
                f"Sub_Types={row['Sub_Types']}\nCell_States={row['Cell_States']}"
            )

        for st, cs in zip(subtypes, states):
            subtype_to_state[st] = (main, cs)

    return RefMaps(main_types=main_types, subtype_to_state=subtype_to_state)

def process_sample_to_spot_json(
    visium_spots,
    predsubclassl1_csv: str,
    predsubclassl2_csv: str,
    cell_subtypes_grouped_csv: str,
) -> Dict[str, Dict[str, Any]]:
    maps = build_reference_maps(cell_subtypes_grouped_csv)

    l1 = _read_celltype_x_spot_csv(predsubclassl1_csv)
    l2 = _read_celltype_x_spot_csv(predsubclassl2_csv)

    spots = [str(spot['properties']['barcode']) for spot in visium_spots['features']]
    common_spots = [c for c in spots if c in set(l1.columns) and c in set(l2.columns)]
    assert set(spots) == set(common_spots), ValueError("We have some spots missing")

    out: Dict[str, Dict[str, Any]] = {}
    for spot, spot_id in zip(visium_spots['features'], spots):
        # --- Main_Cell_Types (from L1) ---
        main_cell_types: Dict[str, float] = {}
        for cell_type, frac in l1[spot_id].items():
            ct = str(cell_type).strip()
            if ct in maps.main_types:
                f = float(frac)
                if f != 0.0:
                    main_cell_types[ct] = f

        # --- Cell_States (from L2) ---
        cell_states: Dict[str, Dict[str, float]] = {}
        for subtype, frac in l2[spot_id].items():
            st = str(subtype).strip()
            if st not in maps.subtype_to_state:
                continue
            main, state = maps.subtype_to_state[st]
            f = float(frac)
            if f == 0.0:
                continue

            cell_states.setdefault(main, {})
            cell_states[main][state] = cell_states[main].get(state, 0.0) + f

        spot['properties'] = spot['properties'] |  {
            "Main_Cell_Types": main_cell_types,
            "Cell_States": cell_states,
        }

    return visium_spots

In [11]:
pred_l1 = os.path.join(data_dir, 'predsubclassl1.csv')
pred_l2 = os.path.join(data_dir, 'predsubclassl2.csv')
ref = os.path.join(data_dir, "Mapping_Materials", "Cell_SubTypes_Grouped.csv")

visium_spots = process_sample_to_spot_json(visium_spots, pred_l1, pred_l2, ref)

histomics_spots = geojson_to_histomics(visium_spots)

gc.post(
    f'/annotation/item/{item}?token={girderToken}',
    data = json.dumps(histomics_spots),
    headers = {
        'X-HTTP-Method': 'POST',
        'Content-Type': 'application/json'
    }
)

1